In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Business Objective

**NLP Sentiment Analysis:** Extracting sentiment from customer reviews on a product.

The project objective requires customer reviews from an e-commerce source such as Amazon and deployment of the final sentiment-analysis result using Streamlit, R Shiny, Flask, or a similar framework.

In [ ]:
df = pd.read_excel('/content/dataset.xlsx')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df['rating'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(6,6))
df['rating'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribution of Customer Ratings')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.show()

In [ ]:
df['rating'].value_counts(normalize=True).sort_index() * 100

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

### Review Text Preparation

The dataset contains the review title, rating, and review body.

For sentiment modelling:

- `title` and `body` are combined into one review text.
- `rating` is used only to create the sentiment target.
- `rating` is **not** given to the machine-learning model, because that would cause target leakage.

In [ ]:
df['title'] = df['title'].fillna('').astype(str)
df['body'] = df['body'].fillna('').astype(str)

df['text'] = (
    df['title'] + ' ' + df['body']
).str.replace(r'\s+', ' ', regex=True).str.strip()

df[['title', 'rating', 'body', 'text']].head()

### Sentiment Label Definition

Because the supplied dataset contains star ratings rather than a separate sentiment column, the target is derived as follows:

| Rating | Sentiment |
|---|---|
| 1–2 | Negative |
| 3 | Neutral |
| 4–5 | Positive |

This rule is kept fixed throughout the project.

In [ ]:
df['sentiment'] = np.select(
    [
        df['rating'].isin([1, 2]),
        df['rating'].eq(3),
        df['rating'].isin([4, 5])
    ],
    [
        'negative',
        'neutral',
        'positive'
    ],
    default='unknown'
)

df['sentiment'].value_counts()

In [ ]:
plt.figure(figsize=(6,6))
df['sentiment'].value_counts().reindex(['negative', 'neutral', 'positive']).plot(kind='bar')
plt.title('Distribution of Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.show()

In [ ]:
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

df[['text_length', 'word_count']].describe()

In [ ]:
plt.figure(figsize=(8,5))
df.boxplot(column='text_length', by='sentiment')
plt.title('Review Length by Sentiment')
plt.suptitle('')
plt.xlabel('Sentiment')
plt.ylabel('Review Length')
plt.show()

### Duplicate Review Check

Repeated review text can cause data leakage if the same review appears in both training and testing data.

Therefore, the original dataset is kept unchanged for reference, while exact duplicate review texts are removed **only for model building**.

In [ ]:
model_df = df[
    (df['text'].str.len() > 0) &
    (df['sentiment'] != 'unknown')
].drop_duplicates(subset=['text']).copy()

model_df.shape

In [ ]:
model_df['sentiment'].value_counts()

### EDA Findings

- The dataset contains customer reviews with title, rating, and review body.
- The target sentiment is derived from the supplied star rating.
- The sentiment classes are negative, neutral, and positive.
- Positive reviews form the largest class, while neutral reviews are the smallest.
- Duplicate review texts were checked and removed from the modelling dataset to reduce the possibility of train/test leakage.
- Rating is used to create the target only and is not used as a model feature.

## Model Building

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df['text']
y = model_df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

### Text Feature Extraction

Machine-learning algorithms cannot directly use raw text.

TF-IDF converts the review text into numerical features.

We use:

- Unigrams: individual words
- Bigrams: two-word phrases
- Sublinear TF scaling

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        max_features=50000
    )),
    ('model', LogisticRegression(
        max_iter=2000,
        class_weight='balanced'
    ))
])

logistic_model.fit(X_train, y_train)

y_pred_logistic = logistic_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_logistic))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_logistic,
    target_names=['Negative', 'Neutral', 'Positive']
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic))

### Multinomial Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

naive_bayes = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        max_features=50000
    )),
    ('model', MultinomialNB())
])

naive_bayes.fit(X_train, y_train)

y_pred_nb = naive_bayes.predict(X_test)

In [ ]:
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=['Negative', 'Neutral', 'Positive']
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

### Linear Support Vector Machine

In [ ]:
from sklearn.svm import LinearSVC

svm_linear = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        max_features=50000
    )),
    ('model', LinearSVC(
        C=1.0,
        class_weight='balanced'
    ))
])

svm_linear.fit(X_train, y_train)

y_pred_svm = svm_linear.predict(X_test)

In [ ]:
print("Linear SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=['Negative', 'Neutral', 'Positive']
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

### Model Comparison

The models are compared using:

- Accuracy
- Macro F1
- Weighted F1

**Macro F1** is important here because the neutral class is smaller than the other two classes. It gives each sentiment class equal importance.

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Multinomial Naive Bayes',
        'Linear SVM'
    ],

    'Accuracy': [
        accuracy_score(y_test, y_pred_logistic),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_svm)
    ],

    'Macro F1': [
        f1_score(y_test, y_pred_logistic, average='macro'),
        f1_score(y_test, y_pred_nb, average='macro'),
        f1_score(y_test, y_pred_svm, average='macro')
    ],

    'Weighted F1': [
        f1_score(y_test, y_pred_logistic, average='weighted'),
        f1_score(y_test, y_pred_nb, average='weighted'),
        f1_score(y_test, y_pred_svm, average='weighted')
    ]
})

results.sort_values(
    by='Macro F1',
    ascending=False
)

### Hyperparameter Tuning for Linear SVM

In [ ]:
from sklearn.model_selection import GridSearchCV

svm = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        max_features=50000
    )),
    ('model', LinearSVC())
])

param_grid = {
    'model__C': [0.1, 0.5, 1, 2, 5],
    'model__class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    svm,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_svm = grid_search.best_estimator_

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Macro F1:")
print(grid_search.best_score_)

In [ ]:
y_pred_best = best_svm.predict(X_test)

print("Tuned SVM Accuracy:", accuracy_score(y_test, y_pred_best))
print("Tuned SVM Macro F1:", f1_score(y_test, y_pred_best, average='macro'))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_best,
    target_names=['Negative', 'Neutral', 'Positive']
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

### Final Model Selection

The final model selected for the project is a **TF-IDF + Linear Support Vector Machine (SVM)**.

The model was selected based on **Macro F1** after comparing Logistic Regression, Multinomial Naive Bayes, and Linear SVM, followed by 5-fold hyperparameter tuning.

The selected configuration from the completed experiment is:

- TF-IDF word unigrams and bigrams
- `C = 2`
- `class_weight = 'balanced'`
- 5-fold GridSearchCV
- Final training on the complete cleaned modelling dataset

The hold-out test accuracy is approximately **79.17%**, with a Macro F1 of approximately **63.70%**.

The neutral class remains the most difficult class to identify, which is an important limitation to mention during the project review.

In [ ]:
final_model = best_svm

In [ ]:
final_model.fit(X, y)

### Sample Prediction

In [ ]:
sample_review = [
    "The product is excellent. The quality is very good and I am completely satisfied."
]

prediction = final_model.predict(sample_review)

prediction

In [ ]:
print("Prediction:", prediction[0].title())

In [ ]:
sample_review = [
    "The product is terrible. The quality is very poor and I regret buying it."
]

prediction = final_model.predict(sample_review)

print("Prediction:", prediction[0].title())

### Save the Final Model

The complete TF-IDF + SVM pipeline is saved as one model file.

This is important because the Streamlit application must use the **same TF-IDF preprocessing and trained classifier** used during training.

In [ ]:
import joblib

joblib.dump(final_model, 'sentiment_model.pkl')

In [ ]:
import os

print(os.listdir())

### Final Project Flow

Customer Review

↓

Title + Review Body

↓

Text Cleaning

↓

TF-IDF

↓

Linear SVM

↓

Sentiment Prediction

↓

Negative / Neutral / Positive

The saved `sentiment_model.pkl` can be loaded directly by the Streamlit application.